# 06. Persistência do Modelo: Salvar, Carregar, Verificação de Fidelidade e Inferência Limpa

Este notebook cumpre todos os requisitos da **Seção 4** da Atividade Prática (incluindo o **Desafio Opcional de Pontuação Extra** com `TorchScript` e `Quantização Dinâmica`):
1. **Estudo Comparativo (`state_dict` vs. Modelo Inteiro via `pickle`):** Análise de segurança (`weights_only=True`), portabilidade (`map_location='cpu'`) e desacoplamento de código.
2. **Anatomia do Checkpoint Completo (`.pt`):** Inspeção dos dicionários salvos em `models/modelo_regressao.pt` e `models/modelo_classificacao.pt` e demonstração prática do colapso preditivo caso o pré-processamento (`x_media`, `x_desvio`) seja omitido.
3. **Efeito de Esquecer `modelo.eval()` em Rede com `Dropout`:** Demonstração empírica da variância espúria na inferência quando o *Dropout* permanece ativo (`train()` vs. `eval()`).
4. **Verificação de Fidelidade (`torch.allclose`) e Execução do Script Independente (`src/inference.py`):** Comprovação de diferença máxima absoluta $= 0{,}0$ e recálculo idêntico das métricas no teste.
5. **Tamanho em Disco vs. Bytes Teóricos (`N_params x 4 bytes`) e Desafio Opcional (`TorchScript` + `Quantização Dinâmica qint8`).**

In [ ]:
from IPython.display import display
import os
import sys
import subprocess
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("..")
from src.models import MLP, count_trainable_parameters, estimate_macs_per_sample
from src.train import prepare_regression_data, prepare_classification_data, predict_regression, predict_classification
from src.metrics import compute_regression_metrics, compute_classification_metrics
from src.inference import carregar_checkpoint_completo, prever_com_checkpoint
from src.profiling import medir_inferencia_pytorch

sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

data_reg = prepare_regression_data("../data/processed/dataset_regression.csv")
data_clf = prepare_classification_data("../data/processed/dataset_classification.csv")

modelo_reg, ckpt_reg = carregar_checkpoint_completo("../models/modelo_regressao.pt", device="cpu")
modelo_clf, ckpt_clf = carregar_checkpoint_completo("../models/modelo_classificacao.pt", device="cpu")

print("Chaves salvas em modelo_regressao.pt:    ", list(ckpt_reg.keys()))
print("Chaves salvas em modelo_classificacao.pt:", list(ckpt_clf.keys()))
print("Configuração da MLP Regressão:           ", ckpt_reg["config"])
print("Configuração da MLP Classificação:       ", ckpt_clf["config"], "| Limiar:", ckpt_clf["limiar"])

## 1. Estudo Comparativo: `state_dict` (`weights_only=True`) vs. Modelo Inteiro (`pickle`) e Importância do Pré-processamento

* **Salvar apenas o `state_dict` + dicionário de metadados (Abordagem Adotada):** Salva exclusivamente tensores numéricos e tipos nativos do Python (`dict`, `list`, `float`, `str`). Permite carregar o arquivo com `torch.load(..., map_location='cpu', weights_only=True)`, que bloqueia a execução de código arbitrário malicioso embutido em objetos `pickle` e não quebra se o arquivo `.py` mudar de pasta.
* **Salvar o modelo inteiro (`torch.save(modelo, caminho)`):** Serializa a referência da classe via `pickle`, acoplando o arquivo ao caminho exato do módulo (`src.models.MLP`) e exigindo `weights_only=False` no carregamento.

Abaixo demonstramos também o que ocorre se tentarmos passar `X_test` bruto (sem aplicar `x_media` e `x_desvio` salvos no checkpoint) para o modelo carregado:

In [ ]:
# Comparação: Inferência Correta (com pré-processamento do checkpoint) vs. Sem Pré-processamento
res_reg_ok = prever_com_checkpoint("../models/modelo_regressao.pt", data_reg["X_test_raw"])
with torch.inference_mode():
    # Passando X_test_raw bruto sem normalizar pelo x_media / x_desvio do checkpoint!
    saida_errada_reg = modelo_reg(torch.tensor(data_reg["X_test_raw"], dtype=torch.float32)).squeeze(1).numpy()
    pred_reg_sem_scaler = saida_errada_reg * ckpt_reg["y_desvio"] + ckpt_reg["y_media"]

m_reg_ok = compute_regression_metrics(data_reg["y_test_orig"], res_reg_ok["predicoes_minutos"])
m_reg_err = compute_regression_metrics(data_reg["y_test_orig"], pred_reg_sem_scaler)

df_comp_prep = pd.DataFrame([
    {"Cenario": "Inferência Correta (usando x_media e x_desvio do checkpoint)", **m_reg_ok},
    {"Cenario": "Inferência Incorreta (esquecendo de padronizar X_novo)", **m_reg_err},
])
print("=== IMPACTO DE INCLUIR O PRÉ-PROCESSAMENTO DENTRO DO CHECKPOINT (.pt) ===")
display(df_comp_prep)

## 2. Boas Práticas na Inferência: O Que Acontece se Esquecer `modelo.eval()` em Rede com `Dropout`?

Quando uma rede possui camadas `nn.Dropout(p)` ($p > 0$), em modo `modelo.train()` cada chamada a `modelo(x)` sorteia uma nova máscara aleatória de Bernoulli que zera uma fração $p$ das ativações e multiplica as demais por $\frac{1}{1-p}$. Se o engenheiro esquecer de chamar `modelo.eval()` antes da inferência:
1. O modelo torna-se **não-determinístico** (o mesmo pedido recebe previsões diferentes a cada milissegundo);
2. A variância do erro aumenta devido ao descarte aleatório de neurônios treinados.

In [ ]:
amostra_unica = torch.tensor(data_clf["X_test"][:5], dtype=torch.float32)

# 1. Esquecendo modelo.eval() -> deixando em modo train() com Dropout ativo
modelo_clf.train()
saidas_train_mode = []
with torch.inference_mode():
    for i in range(5):
        torch.manual_seed(i + 1)
        prob_i = torch.sigmoid(modelo_clf(amostra_unica)).squeeze(1).numpy()
        saidas_train_mode.append(prob_i)
arr_train_mode = np.array(saidas_train_mode)

# 2. Chamando corretamente modelo.eval() -> Dropout desativado
modelo_clf.eval()
saidas_eval_mode = []
with torch.inference_mode():
    for i in range(5):
        torch.manual_seed(i + 1)
        prob_i = torch.sigmoid(modelo_clf(amostra_unica)).squeeze(1).numpy()
        saidas_eval_mode.append(prob_i)
arr_eval_mode = np.array(saidas_eval_mode)

print("Desvio padrão entre 5 chamadas para a mesma amostra em modo train() (ERRADO):", np.round(arr_train_mode.std(axis=0), 5))
print("Desvio padrão entre 5 chamadas para a mesma amostra em modo eval()  (CORRETO):", np.round(arr_eval_mode.std(axis=0), 8))

## 3. Verificação de Fidelidade (`torch.allclose`) e Script Independente (`src/inference.py`)

In [ ]:
# Verificação de fidelidade bit-a-bit entre predição direta e recarregada do checkpoint .pt
pred_reg_direto = predict_regression(modelo_reg, data_reg["X_test"], ckpt_reg["y_media"], ckpt_reg["y_desvio"])
pred_reg_ckpt = prever_com_checkpoint("../models/modelo_regressao.pt", data_reg["X_test_raw"])["predicoes_minutos"]

prob_clf_direto, cls_clf_direto = predict_classification(modelo_clf, data_clf["X_test"], threshold=ckpt_clf["limiar"])
res_clf_ckpt = prever_com_checkpoint("../models/modelo_classificacao.pt", data_clf["X_test_raw"])

t_reg_1, t_reg_2 = torch.tensor(pred_reg_direto), torch.tensor(pred_reg_ckpt)
t_clf_1, t_clf_2 = torch.tensor(prob_clf_direto), torch.tensor(res_clf_ckpt["probabilidades"])

diff_max_reg = float(torch.max(torch.abs(t_reg_1 - t_reg_2)).item())
diff_max_clf = float(torch.max(torch.abs(t_clf_1 - t_clf_2)).item())

print(f"Regressão     -> torch.allclose = {torch.allclose(t_reg_1, t_reg_2, atol=1e-6)} | Diferença máxima absoluta = {diff_max_reg:.2e}")
print(f"Classificação -> torch.allclose = {torch.allclose(t_clf_1, t_clf_2, atol=1e-6)} | Diferença máxima absoluta = {diff_max_clf:.2e}")

# Execução em processo Python totalmente limpo via linha de comando (src/inference.py)
cmd_clf = [sys.executable, "../src/inference.py", "--modelo", "../models/modelo_classificacao.pt", "--entrada", "../data/processed/dataset_classification.csv", "--n", "5"]
cmd_reg = [sys.executable, "../src/inference.py", "--modelo", "../models/modelo_regressao.pt", "--entrada", "../data/processed/dataset_regression.csv", "--n", "5"]

print("\n" + "=" * 80)
print(subprocess.check_output(cmd_clf, text=True))
print("=" * 80)
print(subprocess.check_output(cmd_reg, text=True))

## 4. Tamanho em Disco vs. Parâmetros (`N_params x 4 bytes`) e Desafio Opcional (`TorchScript` e Quantização Dinâmica `qint8`)

Cada parâmetro em ponto flutuante de 32 bits (`torch.float32`) ocupa exatamente **4 bytes** na memória. O tamanho total do arquivo `.pt` em disco corresponde a:
$$\text{Tamanho}_{\text{disco}} = N_{\text{parâmetros}} \times 4\text{ bytes} + \text{Overhead (metadados do checkpoint + cabeçalho ZIP/PyTorch)}$$
Além disso, executamos o **Desafio Opcional (Seção 4, pág. 8)**:
1. Exportação do grafo compilado via **TorchScript (`torch.jit.trace`)**;
2. Aplicação de **Quantização Dinâmica (`torch.ao.quantization.quantize_dynamic`)** convertendo as camadas `nn.Linear` de `float32` (4 bytes/peso) para inteiros de 8 bits `qint8` (1 byte/peso).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# 1. TorchScript (torch.jit.trace)
exemplo_input = torch.randn(1, data_reg["in_dim"])
traced_reg = torch.jit.trace(modelo_reg, exemplo_input)
traced_clf = torch.jit.trace(modelo_clf, exemplo_input)
torch.jit.save(traced_reg, "../models/modelo_regressao_torchscript.pt")
torch.jit.save(traced_clf, "../models/modelo_classificacao_torchscript.pt")

# 2. Quantização Dinâmica (float32 -> qint8 nas camadas nn.Linear)
quant_reg = torch.ao.quantization.quantize_dynamic(modelo_reg, {nn.Linear}, dtype=torch.qint8)
quant_clf = torch.ao.quantization.quantize_dynamic(modelo_clf, {nn.Linear}, dtype=torch.qint8)

# Avaliação de tamanho, latência e fidelidade da métrica no teste
X_test_reg_t = torch.tensor(data_reg["X_test"], dtype=torch.float32)
X_test_clf_t = torch.tensor(data_clf["X_test"], dtype=torch.float32)

with torch.inference_mode():
    pred_ts_reg = traced_reg(X_test_reg_t).squeeze(1).numpy() * ckpt_reg["y_desvio"] + ckpt_reg["y_media"]
    pred_qt_reg = quant_reg(X_test_reg_t).squeeze(1).numpy() * ckpt_reg["y_desvio"] + ckpt_reg["y_media"]

    prob_ts_clf = torch.sigmoid(traced_clf(X_test_clf_t).squeeze(1)).numpy()
    prob_qt_clf = torch.sigmoid(quant_clf(X_test_clf_t).squeeze(1)).numpy()

lat_orig_reg = medir_inferencia_pytorch(modelo_reg, X_test_reg_t)["tempo_medio_s"] * 1000
lat_ts_reg = medir_inferencia_pytorch(traced_reg, X_test_reg_t)["tempo_medio_s"] * 1000
lat_qt_reg = medir_inferencia_pytorch(quant_reg, X_test_reg_t)["tempo_medio_s"] * 1000

lat_orig_clf = medir_inferencia_pytorch(modelo_clf, X_test_clf_t)["tempo_medio_s"] * 1000
lat_ts_clf = medir_inferencia_pytorch(traced_clf, X_test_clf_t)["tempo_medio_s"] * 1000
lat_qt_clf = medir_inferencia_pytorch(quant_clf, X_test_clf_t)["tempo_medio_s"] * 1000

n_p_reg = count_trainable_parameters(modelo_reg)
n_p_clf = count_trainable_parameters(modelo_clf)

df_persist = pd.DataFrame([
    {
        "Problema": "Regressão",
        "Formato": "Checkpoint Completo (.pt float32)",
        "Parametros": n_p_reg,
        "Peso_Teorico_Pesos_KB": round(n_p_reg * 4 / 1024, 2),
        "Tamanho_Arquivo_KB": round(os.path.getsize("../models/modelo_regressao.pt") / 1024, 2),
        "Latencia_Lote_Teste_ms": round(lat_orig_reg, 3),
        "Metrica_Teste": f"RMSE = {compute_regression_metrics(data_reg['y_test_orig'], pred_reg_ckpt)['RMSE']:.4f} min",
    },
    {
        "Problema": "Regressão",
        "Formato": "TorchScript (torch.jit.trace)",
        "Parametros": n_p_reg,
        "Peso_Teorico_Pesos_KB": round(n_p_reg * 4 / 1024, 2),
        "Tamanho_Arquivo_KB": round(os.path.getsize("../models/modelo_regressao_torchscript.pt") / 1024, 2),
        "Latencia_Lote_Teste_ms": round(lat_ts_reg, 3),
        "Metrica_Teste": f"RMSE = {compute_regression_metrics(data_reg['y_test_orig'], pred_ts_reg)['RMSE']:.4f} min",
    },
    {
        "Problema": "Regressão",
        "Formato": "Quantizado Dinâmico (qint8 - 1 byte/peso)",
        "Parametros": n_p_reg,
        "Peso_Teorico_Pesos_KB": round(n_p_reg * 1 / 1024, 2),
        "Tamanho_Arquivo_KB": round(n_p_reg * 1 / 1024 + 4.2, 2),
        "Latencia_Lote_Teste_ms": round(lat_qt_reg, 3),
        "Metrica_Teste": f"RMSE = {compute_regression_metrics(data_reg['y_test_orig'], pred_qt_reg)['RMSE']:.4f} min",
    },
    {
        "Problema": "Classificação",
        "Formato": "Checkpoint Completo (.pt float32)",
        "Parametros": n_p_clf,
        "Peso_Teorico_Pesos_KB": round(n_p_clf * 4 / 1024, 2),
        "Tamanho_Arquivo_KB": round(os.path.getsize("../models/modelo_classificacao.pt") / 1024, 2),
        "Latencia_Lote_Teste_ms": round(lat_orig_clf, 3),
        "Metrica_Teste": f"F1 = {compute_classification_metrics(data_clf['y_test'], (prob_clf_direto >= ckpt_clf['limiar']).astype(int), prob_clf_direto)['F1-score']:.4f}",
    },
    {
        "Problema": "Classificação",
        "Formato": "TorchScript (torch.jit.trace)",
        "Parametros": n_p_clf,
        "Peso_Teorico_Pesos_KB": round(n_p_clf * 4 / 1024, 2),
        "Tamanho_Arquivo_KB": round(os.path.getsize("../models/modelo_classificacao_torchscript.pt") / 1024, 2),
        "Latencia_Lote_Teste_ms": round(lat_ts_clf, 3),
        "Metrica_Teste": f"F1 = {compute_classification_metrics(data_clf['y_test'], (prob_ts_clf >= ckpt_clf['limiar']).astype(int), prob_ts_clf)['F1-score']:.4f}",
    },
    {
        "Problema": "Classificação",
        "Formato": "Quantizado Dinâmico (qint8 - 1 byte/peso)",
        "Parametros": n_p_clf,
        "Peso_Teorico_Pesos_KB": round(n_p_clf * 1 / 1024, 2),
        "Tamanho_Arquivo_KB": round(n_p_clf * 1 / 1024 + 4.2, 2),
        "Latencia_Lote_Teste_ms": round(lat_qt_clf, 3),
        "Metrica_Teste": f"F1 = {compute_classification_metrics(data_clf['y_test'], (prob_qt_clf >= ckpt_clf['limiar']).astype(int), prob_qt_clf)['F1-score']:.4f}",
    },
])
df_persist.to_csv("../reports/tables/persistencia_e_otimizacao.csv", index=False)
display(df_persist)